# Module 6 - Lab 2: Compare Two Measurements of the Same Drive

Two phones recorded the same drive on the same car. Because both were started by hand, one after the other, their clocks do not agree: each recording begins - and ends - at a different moment of the drive.

In this lab you line the two recordings up in time and then evaluate them together.

- How large is the time offset between the two recordings?
- How well do both phones agree once they are aligned?
- Do both measurements lead to the same conclusions about the drive?
- Where do they differ, and what could explain that?

## Learning goals
- Load two related measurements and check that they describe the same quantity.
- Estimate a time offset between two recordings from the data itself.
- Judge an automatically calculated value instead of trusting it blindly.
- Compare two measurements in shared tables and overlaid plots.
- Document which parts of a comparison are measurement and which are your own choice.

<div style="border: 3px solid #b45309; background: #fff7ed; padding: 18px 20px; margin: 16px 0 24px 0; border-radius: 8px;">
<h2 style="margin-top: 0; color: #9a3412;">This lab compares two files directly</h2>
<p>Unlike Lab 1, this notebook does <strong>not</strong> take its dataset from <code>metadata.json</code>. You enter <strong>two file paths</strong> yourself in Section 2, and the sensor type is recognised from the units in the files.</p>
<p>Both files must contain the <strong>same quantity</strong> - either two acceleration recordings or two gyroscope recordings. Mixing them is refused, because their values cannot be compared.</p>
</div>

## Section 1: Import Libraries

In [ ]:
# Section 1: Import Libraries
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd()
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from metadata_loader import load_metadata_context
from analysis_snapshot import declare_lab_inputs
from dual_measurement import (
    apply_time_offset,
    compare_offset_estimates,
    compare_signal_agreement,
    compare_specialized_results,
    compare_time_quality,
    display_time_offset_estimate,
    estimate_time_offset,
    estimate_time_offset_from_timestamps,
    interactive_offset_explorer,
    load_measurement_pair,
    plot_clock_synchronisation,
    plot_pair_axis_overlay,
    plot_pair_overlay,
    plot_time_offset_search,
    plotly_pair_explorer,
    summarize_clock_synchronisation,
    summarize_measurement_pair,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.precision', 4)

print('Libraries imported successfully.')

## Section 2: Choose the Two Measurements

Enter the two files you want to compare. The first one is the reference: it keeps its own time axis, and the second one is the one that gets shifted.

The example paths below use an artificially shifted test file, so the true offset is known and you can check the calculation against it.

In [ ]:
# Section 2: Select the two files
# ============================== YOUR INPUT ==================================
# Both files must contain the same quantity (both acceleration or both gyroscope).
measurement_a_path = 'data/suspension/Example/Beschleunigung ohne g.xls'
measurement_b_path = 'data/suspension/Comparison Example/Phone B acceleration shifted.xlsx'

# Both .csv, .xls/.xlsx and zipped phyphox exports (.zip) can be given here.
# For the gyroscope example, use these two instead:
# measurement_a_path = 'data/suspension/Gyroscope Example/Gyroscope Raw Data.xls'
# measurement_b_path = 'data/suspension/Comparison Example/Phone B gyroscope shifted.xlsx'

measurement_a_label = 'phone A'
measurement_b_label = 'phone B'
# ============================================================================

In [ ]:
# Section 2: Load both measurements
metadata = load_metadata_context(project_root)['public_metadata']

measurement_pair = load_measurement_pair(
    measurement_a_path,
    measurement_b_path,
    metadata,
    project_root=project_root,
    label_a=measurement_a_label,
    label_b=measurement_b_label,
)

print('Recognised quantity:', measurement_pair['quantity'])
print('Analysis mode      :', measurement_pair['analysis_key'])
display(summarize_measurement_pair(measurement_pair))

### Observation 1: Before Alignment

Look at the start and end times of both recordings above.

- Which phone was started first, and by roughly how much?
-

## Section 3: Data Quality of Both Recordings

Before comparing results, check that both recordings are equally usable. A large difference in sampling rate or gaps in one of them would weaken every later comparison.

In [ ]:
# Section 3: Time step quality side by side
display(compare_time_quality(measurement_pair))

In [ ]:
# Section 3: Both signals before any alignment
plot_pair_overlay(measurement_pair)

## Section 4: Find the Time Offset

There are two independent ways to find out how far the recordings are apart, and this lab uses both.

The **signals themselves** can be shifted against each other until they agree best. For this comparison only, both are interpolated onto a shared time grid, because the two phones do not sample at the same moments. **This interpolation is used for the offset search only.** Every result later in this notebook is calculated from the originally recorded samples; the chosen offset merely moves the time axis of the second measurement.

The **clocks of the phones** offer a second answer. phyphox stores the wall-clock moment at which each recording was started, so the difference between those two moments is the offset as well - without looking at a single measured value.

Do not expect the two answers to be identical. Where they differ, that difference is itself a measurement result, and Section 4b works out what it tells you. Both values are only suggestions; neither is applied automatically.

In [ ]:
# Section 4: Calculate the offset from the signals
offset_estimate = estimate_time_offset(measurement_pair, max_offset_s=30.0)

display(display_time_offset_estimate(offset_estimate, measurement_pair))

### Second Opinion: The Clocks in the Files

The table below puts both answers next to each other. They come from completely independent sources - the measured values on one side, the phone clocks on the other.

Watch the third row. If it is not close to zero, the optimum found in the signals does **not** correspond to the difference between the two clocks.

In [ ]:
# Section 4: Compare the signal-based and the clock-based offset
timestamp_estimate = estimate_time_offset_from_timestamps(measurement_pair)

display(compare_offset_estimates(measurement_pair, offset_estimate, timestamp_estimate))
plot_time_offset_search(offset_estimate, timestamp_estimate)

## Section 4b: Are the Two Phone Clocks Synchronised?

The signals show when the drive really happened. The timestamps show when the phones *believed* it happened. If the two disagree, the phones are right about the drive and wrong about the time - their clocks are not synchronised.

This is worth pausing on: a single recording can never reveal this. Its timestamps look perfectly plausible on their own. The error only becomes measurable because a second phone recorded the same drive, and the physical signal provides a reference the clocks can be checked against.

The plot shows both alignments directly. If the clocks were wrong, the lower panel is visibly out of step while the upper one is not - even though both were produced from exactly the same measurements.

In [ ]:
# Section 4b: Check the phone clocks against the measured signals
display(summarize_clock_synchronisation(measurement_pair, offset_estimate, timestamp_estimate))
plot_clock_synchronisation(measurement_pair, offset_estimate, timestamp_estimate)

### Observation 2: The Phone Clocks

Document what the comparison showed, before you choose an offset.

- Which offset did the signals suggest, and which one did the clocks suggest?
- How far apart are the two, and which phone's clock is ahead?
- How large would the misalignment be if you had trusted the timestamps? Is that a lot compared to the events in your data?
- Which of the two sources do you trust for this dataset, and why?
- What would you have to do before a timestamp could be trusted for alignment?
-

Both values above are **suggestions**, not settings. Nothing has been applied to the data yet.

### Explore: Shift the Measurements Yourself

Move the slider and watch how the two curves line up. The slider starts at zero, so you can see the effect of the offset before you decide which value to use.

In [ ]:
# Section 4: Explore the offset interactively
interactive_offset_explorer(measurement_pair, max_offset_s=30.0)

### Choose the Offset to Work With

Now enter the offset that the rest of the notebook should use. You can take the calculated value, a value you found with the slider, or a value you have another reason for. Whatever you enter here is what every following result is based on, so write down why you chose it.

In [ ]:
# Section 4: Set the offset used for all further results
# ============================== YOUR INPUT ==================================
# Seconds added to the time axis of measurement B. 0.0 means no correction.
# The calculated suggestion is printed above - decide whether you follow it.
time_offset_seconds = 0.0
# ============================================================================

In [ ]:
# Section 4: Apply the chosen offset
aligned_pair = apply_time_offset(measurement_pair, time_offset_seconds)

print('Calculated suggestion:', round(offset_estimate['offset_seconds'], 4), 's')
print('Your chosen offset   :', time_offset_seconds, 's')
print('Difference           :', round(time_offset_seconds - offset_estimate['offset_seconds'], 4), 's')

display(compare_signal_agreement(aligned_pair))
plot_pair_overlay(aligned_pair)

In [ ]:
# Section 4: Inspect the alignment interactively (zoom, hover, toggle traces)
plotly_pair_explorer(aligned_pair)

### Observation 3: Your Offset Choice

- Which offset did you use, and why?
- How close is it to the value the signals suggested?
- Does the correlation confirm your choice, or did the slider look better than the number suggests?
-

## Section 5: Compare the Results

Both measurements are now evaluated exactly as in Lab 1: each axis is smoothed and the mode-specific result is calculated. Because both recordings show the same drive, the two result columns should be close to each other - and where they are not, that difference is the interesting part.

In [ ]:
# Section 5: All axes of both measurements above one another
plot_pair_axis_overlay(aligned_pair)

In [ ]:
# Section 5: Mode-specific results next to each other
display(compare_specialized_results(aligned_pair, metadata))

### Observation 4: Agreement and Differences

- Which values agree well between the two phones?
- Which differ most, and by how much?
- Is the remaining difference explained by the mounting position, by the sensors themselves, or by your offset choice?
- Would both phones lead you to the same conclusion about this drive?
-

### What This Run Used

The export in Module 10 does not guess which files an analysis was based on. It reads the declaration below, and refuses to build a package without it. That way the resulting RO-Crate can always state what it was made from.

In [ ]:
# Section 6: Record what this run used
lab_inputs = declare_lab_inputs(
    notebook='lab_06_2_compare_two_measurements_jupyter.ipynb',
    measurement_files=[measurement_a_path, measurement_b_path],
    analysis_choices={
        'time_offset_seconds': time_offset_seconds,
        'calculated_offset_seconds': round(offset_estimate['offset_seconds'], 4),
        'measurement_a_label': measurement_a_label,
        'measurement_b_label': measurement_b_label,
    },
    project_root=project_root,
)
display(pd.json_normalize(lab_inputs, sep='.').T.rename(columns={0: 'value'}))

## Section 6: Document Your Comparison

Summarise what this comparison supports and what it does not.

**What I compared:**

**Offset I used and why:**

**Where both measurements agree:**

**Where they disagree, and my explanation:**

**What I would need to decide this properly:**